# 1. Data Processing - UNSW-NB15


In [1]:
import os, random, json, warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.preprocessing import LabelEncoder as ColEncoder

import joblib

warnings.filterwarnings('ignore', category=RuntimeWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

raw_dir = os.path.join('..', 'data', 'raw')
splits_dir = os.path.join('..', 'data', 'splits')
models_dir = os.path.join('..', 'models')

os.makedirs(splits_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)

print(raw_dir, splits_dir, models_dir)

..\data\raw ..\data\splits ..\models


In [2]:
files = os.listdir(raw_dir)
train_candidates = [f for f in files if 'UNSW_NB15_training-set' in f and f.endswith('.csv')]
test_candidates = [f for f in files if 'UNSW_NB15_testing-set' in f and f.endswith('.csv')]

if not train_candidates or not test_candidates:
    raise FileNotFoundError('Put UNSW_NB15_training-set.csv and UNSW_NB15_testing-set.csv in data/raw')

train_df_raw = pd.read_csv(os.path.join(raw_dir, train_candidates[0]))
test_df_raw = pd.read_csv(os.path.join(raw_dir, test_candidates[0]))

train_df_raw['split'] = 'train'
test_df_raw['split'] = 'test'

df = pd.concat([train_df_raw, test_df_raw], ignore_index=True)
df.columns = df.columns.str.strip().str.lower()

print(df.shape)
print(df.columns.tolist())

(257673, 46)
['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label', 'split']


In [3]:
for c in ['id', 'srcip', 'dstip']:
    if c in df.columns:
        df = df.drop(columns=[c])

ac = df['attack_cat'].astype(str).str.strip().str.strip("'\"")
ac = ac.replace({'': 'Normal', 'nan': 'Normal', 'NaN': 'Normal'})
ac = ac.fillna('Normal').replace({
    'Backdoors': 'Backdoor',
    'backdoor': 'Backdoor',
    'backdoors': 'Backdoor',
    'normal': 'Normal'
})
df['attack_cat'] = ac

print(df['attack_cat'].value_counts())

attack_cat
Normal            93000
Generic           58871
Exploits          44525
Fuzzers           24246
DoS               16353
Reconnaissance    13987
Analysis           2677
Backdoor           2329
Shellcode          1511
Worms               174
Name: count, dtype: int64


In [4]:
for col in df.columns:
    if df[col].dtype == 'object' and col != 'attack_cat':
        if df[col].isna().any():
            df[col] = df[col].fillna(df[col].mode().iloc[0])
    elif df[col].dtype != 'object':
        if df[col].isna().any():
            df[col] = df[col].fillna(df[col].median())

df = df.replace([np.inf, -np.inf], np.nan)
for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

print('Remaining NaNs:', int(df.isna().sum().sum()))

Remaining NaNs: 0


In [5]:
if 'proto' in df.columns:
    pe = ColEncoder()
    df['proto'] = pe.fit_transform(df['proto'].astype(str))
    joblib.dump(pe, os.path.join(models_dir, 'proto_encoder.pkl'))

if 'service' in df.columns:
    top_service = df['service'].value_counts().nlargest(15).index
    df['service'] = df['service'].where(df['service'].isin(top_service), other='other')
    df = pd.get_dummies(df, columns=['service'], drop_first=False)

if 'state' in df.columns:
    df = pd.get_dummies(df, columns=['state'], drop_first=False)

print(df.shape)

(257673, 67)


In [6]:
def coerce_to_numeric(series):
    def parse(v):
        try:
            if isinstance(v, str) and v.startswith('0x'):
                return int(v, 16)
            return float(v)
        except (ValueError, TypeError):
            return np.nan
    return series.apply(parse)

protected = {'attack_cat', 'label', 'split'}
obj_cols = [c for c in df.select_dtypes(include='object').columns if c not in protected]

for col in obj_cols:
    df[col] = coerce_to_numeric(df[col])
    if df[col].isna().all():
        df[col] = 0.0
    else:
        df[col] = df[col].fillna(df[col].median())

print('Remaining object columns:', df.select_dtypes(include='object').columns.tolist())

Remaining object columns: ['attack_cat', 'split']


In [7]:
extra_drop = ['split'] + (['label'] if 'label' in df.columns else [])
features_df = df.drop(columns=['attack_cat'] + extra_drop)

le = LabelEncoder()
y_all = le.fit_transform(df['attack_cat'])
joblib.dump(le, os.path.join(models_dir, 'label_encoder.pkl'))

with open(os.path.join(models_dir, 'class_mapping.json'), 'w') as f:
    json.dump({int(i): c for i, c in enumerate(le.classes_)}, f, indent=2)

mask_train = df['split'].eq('train').values
mask_test = df['split'].eq('test').values

X_train_full = features_df[mask_train].copy()
y_train_full = y_all[mask_train]

X_test_full = features_df[mask_test].copy()
y_test_full = y_all[mask_test]

print(X_train_full.shape, X_test_full.shape)
print(le.classes_)

(175341, 64) (82332, 64)
['Analysis' 'Backdoor' 'DoS' 'Exploits' 'Fuzzers' 'Generic' 'Normal'
 'Reconnaissance' 'Shellcode' 'Worms']


In [8]:
X_train_base, X_val_base, y_train_base, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=SEED
)

print(X_train_base.shape, X_val_base.shape)

(140272, 64) (35069, 64)


In [9]:
et = ExtraTreesClassifier(
    n_estimators=300,
    random_state=SEED,
    n_jobs=-1,
    class_weight='balanced'
)
et.fit(X_train_base, y_train_base)

importances = et.feature_importances_
feat_names = np.array(X_train_base.columns)

idx = np.argsort(importances)[::-1][:30]
selected_features = feat_names[idx].tolist()

joblib.dump(selected_features, os.path.join(models_dir, 'selected_features.pkl'))

expert_feature_sets = {
    'expert1': selected_features[:15],
    'expert2': selected_features[:30],
    'expert3': selected_features[:30]
}
joblib.dump(expert_feature_sets, os.path.join(models_dir, 'expert_feature_sets.pkl'))

print('Selected features:', selected_features)
print('Expert feature sets:', {k: len(v) for k, v in expert_feature_sets.items()})

Selected features: ['smean', 'sttl', 'sbytes', 'service_-', 'service_dns', 'ct_srv_dst', 'ct_dst_src_ltm', 'ct_srv_src', 'sload', 'service_http', 'dttl', 'ct_src_dport_ltm', 'proto', 'rate', 'ct_dst_sport_ltm', 'dmean', 'ct_src_ltm', 'ct_dst_ltm', 'dur', 'dload', 'sinpkt', 'ct_state_ttl', 'state_INT', 'sjit', 'dbytes', 'synack', 'tcprtt', 'dpkts', 'stcpb', 'dtcpb']
Expert feature sets: {'expert1': 15, 'expert2': 30, 'expert3': 30}


In [10]:
X_train_sel = X_train_base[selected_features].copy()
X_val_sel = X_val_base[selected_features].copy()
X_test_sel = X_test_full[selected_features].copy()

for X in [X_train_sel, X_val_sel, X_test_sel]:
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    for c in X.columns:
        if X[c].isna().all():
            X[c] = 0.0
        elif X[c].isna().any():
            X[c] = X[c].fillna(X[c].median())

print('NaNs train:', int(X_train_sel.isna().sum().sum()))
print('NaNs val  :', int(X_val_sel.isna().sum().sum()))
print('NaNs test :', int(X_test_sel.isna().sum().sum()))

NaNs train: 0
NaNs val  : 0
NaNs test : 0


In [11]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(sampling_strategy='not majority', random_state=SEED)
X_train_os, y_train_os = ros.fit_resample(X_train_sel, y_train_base)

print('Base train shape:', X_train_sel.shape)
print('Oversampled shape:', X_train_os.shape)
print('Before:', dict(zip(*np.unique(y_train_base, return_counts=True))))
print('After :', dict(zip(*np.unique(y_train_os, return_counts=True))))

Base train shape: (140272, 30)
Oversampled shape: (448000, 30)
Before: {np.int64(0): np.int64(1600), np.int64(1): np.int64(1397), np.int64(2): np.int64(9811), np.int64(3): np.int64(26714), np.int64(4): np.int64(14547), np.int64(5): np.int64(32000), np.int64(6): np.int64(44800), np.int64(7): np.int64(8393), np.int64(8): np.int64(906), np.int64(9): np.int64(104)}
After : {np.int64(0): np.int64(44800), np.int64(1): np.int64(44800), np.int64(2): np.int64(44800), np.int64(3): np.int64(44800), np.int64(4): np.int64(44800), np.int64(5): np.int64(44800), np.int64(6): np.int64(44800), np.int64(7): np.int64(44800), np.int64(8): np.int64(44800), np.int64(9): np.int64(44800)}


In [12]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_os)
X_val_s = scaler.transform(X_val_sel)
X_test_s = scaler.transform(X_test_sel)

joblib.dump(scaler, os.path.join(models_dir, 'scaler.pkl'))

classes = np.unique(y_train_base)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_base)
weights = np.sqrt(weights)
weights = weights / weights.mean()
weights = np.clip(weights, 0.1, 5.0)
np.save(os.path.join(models_dir, 'class_weights.npy'), weights)

train_df = pd.DataFrame(X_train_s, columns=selected_features)
train_df['label'] = y_train_os

val_df = pd.DataFrame(X_val_s, columns=selected_features)
val_df['label'] = y_val

test_df = pd.DataFrame(X_test_s, columns=selected_features)
test_df['label'] = y_test_full

train_df.to_csv(os.path.join(splits_dir, 'train.csv'), index=False)
val_df.to_csv(os.path.join(splits_dir, 'val.csv'), index=False)
test_df.to_csv(os.path.join(splits_dir, 'test.csv'), index=False)

print('Saved train/val/test and artifacts')

Saved train/val/test and artifacts
